# Word Embeddings — Word2Vec & GloVe

> Based on Stanford CS230 — C5M2, Andrew Ng / DeepLearning.AI

One-hot vectors have no notion of word similarity. **Word embeddings** map words into a dense vector space where semantically related words are geometrically close — enabling transfer learning for NLP.

## Learning Objectives

1. Explain why one-hot encodings fail to capture word similarity
2. Describe the Skip-gram Word2Vec objective
3. Understand the GloVe objective using co-occurrence statistics
4. Explore analogies via vector arithmetic: king − man + woman ≈ queen

## Word Analogies

$$\text{Find } w : \arg\max_{w}\; \text{sim}\!\bigl(e_w,\; e_{\text{king}} - e_{\text{man}} + e_{\text{woman}}\bigr)$$

Typical similarity measure: cosine similarity

$$\text{sim}(u, v) = \frac{u \cdot v}{\|u\|\|v\|}$$

## Word2Vec Skip-gram Objective

$$\max \sum_{t} \sum_{-c \le j \le c, j \ne 0} \log P(w_{t+j} \mid w_t)$$

$$P(w_O \mid w_I) = \frac{\exp(v_{w_O}^{\top}\, u_{w_I})}{\sum_{w=1}^{V} \exp(v_w^{\top}\, u_{w_I})}$$

## GloVe Objective

$$J = \sum_{i,j=1}^{V} f(X_{ij}) \bigl(\theta_i^{\top} e_j + b_i + b_j - \log X_{ij}\bigr)^2$$

where $X_{ij}$ is the co-occurrence count and $f$ is a weighting function that down-weights rare and very frequent pairs.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

def cosine_sim(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-10)

# ---- Synthetic 2-D embeddings (illustrative) ----
# We'll create a small embedding space that captures gender and royalty
# Each word has a 2D embedding [royalty_dim, gender_dim]
words = ['man', 'woman', 'king', 'queen', 'boy', 'girl', 'prince', 'princess',
         'apple', 'orange', 'mango', 'banana']
embeddings_2d = np.array([
    [-0.1,  -0.8],  # man
    [-0.1,   0.8],  # woman
    [ 0.9,  -0.8],  # king
    [ 0.9,   0.8],  # queen
    [-0.3,  -0.5],  # boy
    [-0.3,   0.5],  # girl
    [ 0.6,  -0.6],  # prince
    [ 0.6,   0.6],  # princess
    [-1.5,   0.1],  # apple
    [-1.5,  -0.2],  # orange
    [-1.7,   0.2],  # mango
    [-1.6,  -0.3],  # banana
])

word_idx = {w: i for i, w in enumerate(words)}

def analogy(a, b, c, embeddings, word_idx, words):
    """Find d such that a:b :: c:d"""
    ea, eb, ec = embeddings[word_idx[a]], embeddings[word_idx[b]], embeddings[word_idx[c]]
    target = eb - ea + ec
    sims = [(cosine_sim(target, embeddings[i]), w)
            for i, w in enumerate(words) if w not in {a, b, c}]
    return sorted(sims, reverse=True)[:3]

print("Word Analogy:  man : king  ::  woman : ?")
results = analogy('man', 'king', 'woman', embeddings_2d, word_idx, words)
for sim, word in results:
    print(f"  {word:12s}  cosine sim = {sim:.4f}")

print("\nWord Analogy:  boy : girl  ::  prince : ?")
results2 = analogy('boy', 'girl', 'prince', embeddings_2d, word_idx, words)
for sim, word in results2:
    print(f"  {word:12s}  cosine sim = {sim:.4f}")

# ---- Visualise embedding space ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Word Embeddings — 2D Visualisation', fontsize=13, fontweight='bold')

color_map = {
    'man': P[0], 'king': P[0], 'boy': P[0], 'prince': P[0],
    'woman': P[1], 'queen': P[1], 'girl': P[1], 'princess': P[1],
    'apple': P[2], 'orange': P[2], 'mango': P[2], 'banana': P[2],
}

ax = axes[0]
for word, emb in zip(words, embeddings_2d):
    ax.scatter(*emb, color=color_map[word], s=80, zorder=3)
    ax.annotate(word, emb, textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.axhline(0, color='#c8ccd4', lw=1); ax.axvline(0, color='#c8ccd4', lw=1)
ax.set_xlabel('royalty dimension'); ax.set_ylabel('gender dimension')
ax.set_title('Word vectors in 2D space')
ax.grid(True)
# Analogy arrow: man→king, woman→queen
ax.annotate('', xy=embeddings_2d[word_idx['king']],
            xytext=embeddings_2d[word_idx['man']],
            arrowprops=dict(arrowstyle='->', color=P[5], lw=2))
ax.annotate('', xy=embeddings_2d[word_idx['queen']],
            xytext=embeddings_2d[word_idx['woman']],
            arrowprops=dict(arrowstyle='->', color=P[5], lw=2, ls='dashed'))
ax.text(0.35, -0.05, 'royalty\nvector', color=P[5], fontsize=8, ha='center')

# ---- Cosine similarity heatmap ----
ax = axes[1]
sample = ['man','woman','king','queen','apple','orange']
idx    = [word_idx[w] for w in sample]
mat    = np.array([[cosine_sim(embeddings_2d[i], embeddings_2d[j]) for j in idx] for i in idx])
im = ax.imshow(mat, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(sample))); ax.set_yticks(range(len(sample)))
ax.set_xticklabels(sample, rotation=45, ha='right')
ax.set_yticklabels(sample)
ax.set_title('Cosine Similarity Matrix')
for i in range(len(sample)):
    for j in range(len(sample)):
        ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='black' if abs(mat[i,j]) < 0.7 else 'white')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()
